<a href="https://colab.research.google.com/github/arog-bioinfo/SIB-Group-Project/blob/genomic_module/notebooks/05_ESM2_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Language Models for Spike Mutation Prediction

**Objective**: Generate rich biological representations of spike protein sequences using ESM-2, then use embeddings to predict spike mutation counts with XGBoost.

***

## 1. Setup & Install ESM-2

### 1.1 Load required Libraries

In [1]:
import subprocess
import sys
import pandas as pd
import numpy as np
import torch
from typing import List

# ESM-2 and ML
try:
    import esm
    print("ESM already installed")
except ImportError:
    print("Installing ESM...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fair-esm"])
    import esm

# ML libraries
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error
import joblib
import pickle
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")
print(f"PyTorch GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


Installing ESM...
All imports successful
PyTorch GPU: True
Device: Tesla T4


## 2. Load Data & ESM-2 Model


In [3]:
# Load dataset
df = pd.read_csv('final_dataset_with_spike.csv')
print(f"Dataset shape: {df.shape}")
print(f"Spike sequences: {df['spike_protein_sequence'].notna().sum()} / {len(df)}")

# Load ESM-2 (650M params)
print("\nLoading ESM-2...")
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
    device = 'cuda'
else:
    device = 'cpu'
print(f"ESM-2 loaded on {device}")


Dataset shape: (3500, 13)
Spike sequences: 3021 / 3500

Loading ESM-2...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
ESM-2 loaded on cuda


## 3. Generate ESM-2 Embeddings

In [4]:
def clean_for_esm(seq):
    """Remove stop codons and invalid characters for ESM-2."""
    if pd.isna(seq):
        return ""
    seq = str(seq).replace('*', 'X').replace('**', 'X')
    valid_aa = 'ACDEFGHIKLMNPQRSTVWY'
    seq = ''.join([c for c in seq if c in valid_aa])
    return seq if len(seq) >= 1200 else ""

def generate_esm_embeddings(sequences: List[str], batch_size=32, max_len=1024):
    """Generate 1280-dim ESM-2 embeddings (mean-pooled)."""
    batch_converter = alphabet.get_batch_converter()
    embeddings_list = []
    device = next(model.parameters()).device

    for i in range(0, len(sequences), batch_size):
        batch_seqs = [str(s) for s in sequences[i:i+batch_size] if len(str(s)) > 0]
        if len(batch_seqs) == 0:
            continue

        batch_data = [(f"seq_{j}", seq[:max_len]) for j, seq in enumerate(batch_seqs)]

        try:
            batch_labels, batch_strs, batch_tokens = batch_converter(batch_data)
            batch_tokens = batch_tokens.to(device)

            with torch.no_grad():
                results = model(batch_tokens, repr_layers=[33], return_contacts=False)
                embeddings = results["representations"][33]

            mean_embeddings = embeddings.mean(dim=1).cpu().numpy()
            embeddings_list.append(mean_embeddings)

            if (i // batch_size + 1) % 10 == 0:
                print(f"  Processed {i + len(batch_seqs):,} / {len(sequences):,}")

        except Exception as e:
            print(f"  Error at {i}: {e}")
            embeddings_list.append(np.zeros((len(batch_seqs), 1280)))

    return np.vstack(embeddings_list) if embeddings_list else np.zeros((len(sequences), 1280))

# Clean sequences and generate embeddings
print("Cleaning spike sequences...")
cleaned_seqs = df['spike_protein_sequence'].apply(clean_for_esm)
valid_seqs = cleaned_seqs[cleaned_seqs.str.len() > 0].tolist()
print(f"Valid sequences: {len(valid_seqs)} / {len(df)}")

print(f"\nGenerating ESM-2 embeddings...")
embeddings = generate_esm_embeddings(valid_seqs, batch_size=32 if torch.cuda.is_available() else 8)
print(f"Embeddings shape: {embeddings.shape}")

# Pad to full dataset size
if embeddings.shape[0] < len(df):
    pad = np.zeros((len(df) - embeddings.shape[0], embeddings.shape[1]))
    embeddings = np.vstack([embeddings, pad])

embeddings_df = pd.DataFrame(embeddings, columns=[f"esm2_dim_{i}" for i in range(embeddings.shape[1])])
print(f"Embeddings aligned: {embeddings_df.shape}")

Cleaning spike sequences...
Valid sequences: 1857 / 3500

Generating ESM-2 embeddings...
  Processed 320 / 1,857
  Processed 640 / 1,857
  Processed 960 / 1,857
  Processed 1,280 / 1,857
  Processed 1,600 / 1,857
Embeddings shape: (1857, 1280)
Embeddings aligned: (3500, 1280)



## 4. Temporal Train/Test Split

In [5]:
# Temporal split: 80% train, 20% test
df['days_since_start'] = pd.to_datetime(df['collection_date']).apply(
    lambda x: (x - pd.to_datetime('2020-01-28')).days
)

df_sorted = df.sort_values('days_since_start').reset_index(drop=True)
cutoff = df_sorted['days_since_start'].quantile(0.8)
train_idx = df_sorted['days_since_start'] <= cutoff

print(f"Train: {train_idx.sum()} | Test: {(~train_idx).sum()}")
print(f"Train days: {df_sorted.loc[train_idx, 'days_since_start'].min():.0f} - {df_sorted.loc[train_idx, 'days_since_start'].max():.0f}")
print(f"Test days: {df_sorted.loc[~train_idx, 'days_since_start'].min():.0f} - {df_sorted.loc[~train_idx, 'days_since_start'].max():.0f}")


Train: 2808 | Test: 692
Train days: 31 - 820
Test days: 821 - 1326


## 5. Data Filtering: Remove Data Shift Issues

In [6]:
# Issue: Test set has new clades (XBB.1.5) with different mutation distribution
# Solution: Train/test on shared clades only

clados_train = set(df_sorted.loc[train_idx, 'phenotype_clade'].unique())
clados_test = set(df_sorted.loc[~train_idx, 'phenotype_clade'].unique())
shared_clados = clados_train & clados_test

print(f"Train clades: {clados_train}")
print(f"Test clades: {clados_test}")
print(f"Shared: {shared_clados}")
print(f"Test exclusive (new): {clados_test - clados_train}")

# Filter to shared clades
train_mask_shared = (df_sorted.loc[train_idx, 'phenotype_clade'].isin(shared_clados)).values
test_mask_shared = (df_sorted.loc[~train_idx, 'phenotype_clade'].isin(shared_clados)).values

df_train = df_sorted.loc[train_idx].iloc[train_mask_shared]
df_test = df_sorted.loc[~train_idx].iloc[test_mask_shared]

y_train = df_train['phenotype_number spike muts from Wuhan-Hu-1'].values
y_test = df_test['phenotype_number spike muts from Wuhan-Hu-1'].values

print(f"\nShared clades subset:")
print(f"  Train: n={len(y_train)}, mean={y_train.mean():.1f}, std={y_train.std():.1f}")
print(f"  Test: n={len(y_test)}, mean={y_test.mean():.1f}, std={y_test.std():.1f}")

# Corresponding feature matrices
X_train_emb = embeddings_df.iloc[df_train.index].values
X_test_emb = embeddings_df.iloc[df_test.index].values


Train clades: {'BA.1', 'B.1', 'BA.2', 'B.1.351'}
Test clades: {'XBB.1.5', 'BA.2', 'B.1.351', 'BA.1'}
Shared: {'BA.1', 'BA.2', 'B.1.351'}
Test exclusive (new): {'XBB.1.5'}

Shared clades subset:
  Train: n=2308, mean=27.5, std=9.4
  Test: n=192, mean=30.9, std=1.6



## 6. PCA + Features + One-Hot Encoding

In [7]:
# PCA on embeddings
scaler_pca = StandardScaler()
X_train_emb_scaled = scaler_pca.fit_transform(X_train_emb)
X_test_emb_scaled = scaler_pca.transform(X_test_emb)

pca = PCA(n_components=0.95)
X_train_emb_pca = pca.fit_transform(X_train_emb_scaled)
X_test_emb_pca = pca.transform(X_test_emb_scaled)

print(f"PCA: 1280 → {pca.n_components_} components (95% variance)")

# Categorical features
feature_cols = ['days_since_start', 'phenotype_clade', 'geographic_location']
oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = oh.fit_transform(df_train[feature_cols])
X_test_cat = oh.transform(df_test[feature_cols])

# Combine
X_train_full = np.hstack([X_train_emb_pca, X_train_cat])
X_test_full = np.hstack([X_test_emb_pca, X_test_cat])

print(f"Final features: {X_train_full.shape} (PCA: {pca.n_components_}, Categorical: {X_train_cat.shape})")


PCA: 1280 → 2 components (95% variance)
Final features: (2308, 642) (PCA: 2, Categorical: (2308, 640))


## 7. Cross-Validation + XGBoost Training

In [8]:
# Temporal cross-validation
tscv = TimeSeriesSplit(n_splits=5)

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    reg_alpha=1,
    reg_lambda=1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# Cross-validation
cv_scores = cross_val_score(xgb_model, X_train_full, y_train, cv=tscv, scoring='r2')
print(f"CV R² (Temporal): {cv_scores.mean():.3f} ± {cv_scores.std()*2:.3f}")

# Fit on full train set
xgb_model.fit(X_train_full, y_train)

# Evaluate
y_pred_train = xgb_model.predict(X_train_full)
y_pred_test = xgb_model.predict(X_test_full)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)

print(f"\nResults:")
print(f"  Train R²: {r2_train:.3f}")
print(f"  Test R²: {r2_test:.3f}")
print(f"  Test MAE: {mae_test:.3f}")


CV R² (Temporal): 0.548 ± 1.801

Results:
  Train R²: 1.000
  Test R²: 1.000
  Test MAE: 0.009


## 8. Feature Importance Analysis

In [9]:
importances = xgb_model.feature_importances_
top_20_idx = np.argsort(importances)[-20:][::-1]

print(f"Top 20 Features:")
for rank, idx in enumerate(top_20_idx, 1):
    if idx < pca.n_components_:
        feat_name = f"ESM-PCA_{idx}"
    else:
        feat_name = f"Categorical_{idx - pca.n_components_}"
    print(f"  {rank:2d}. {feat_name:20s} importance={importances[idx]:.5f}")

# Summary
esm_imp = importances[:pca.n_components_].sum()
cat_imp = importances[pca.n_components_:].sum()
total_imp = esm_imp + cat_imp

print(f"\nSummary:")
print(f"  ESM-2 PCA: {esm_imp:.3f} ({esm_imp/total_imp*100:.1f}%)")
print(f"  Categorical: {cat_imp:.3f} ({cat_imp/total_imp*100:.1f}%)")


Top 20 Features:
   1. Categorical_481      importance=0.75462
   2. Categorical_483      importance=0.14607
   3. ESM-PCA_0            importance=0.05088
   4. Categorical_482      importance=0.04305
   5. ESM-PCA_1            importance=0.00226
   6. Categorical_518      importance=0.00090
   7. Categorical_363      importance=0.00050
   8. Categorical_635      importance=0.00049
   9. Categorical_205      importance=0.00031
  10. Categorical_638      importance=0.00018
  11. Categorical_214      importance=0.00014
  12. Categorical_360      importance=0.00012
  13. Categorical_549      importance=0.00010
  14. Categorical_523      importance=0.00009
  15. Categorical_208      importance=0.00009
  16. Categorical_636      importance=0.00007
  17. Categorical_540      importance=0.00006
  18. Categorical_361      importance=0.00002
  19. Categorical_246      importance=0.00001
  20. Categorical_471      importance=0.00001

Summary:
  ESM-2 PCA: 0.053 (5.3%)
  Categorical: 0.947 (94.7%

## 9. Save Model for Deployment

In [10]:
# Save trained model + preprocessing
model_artifact = {
    'model': xgb_model,
    'pca': pca,
    'scaler_pca': scaler_pca,
    'onehotencoder': oh,
    'feature_cols': feature_cols,
    'trained_clades': list(shared_clados),
    'performance': {
        'cv_r2': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
        'test_r2': r2_test,
        'test_mae': mae_test,
        'train_r2': r2_train,
        'n_train': len(y_train),
        'n_test': len(y_test)
    }
}

with open('spike_mutations_predictor.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print("Model saved: spike_mutations_predictor.pkl")
print(f"\nModel Info:")
print(f"  Clades: {list(shared_clados)}")
print(f"  Performance: R² = {r2_test:.3f}, MAE = {mae_test:.3f}")


Model saved: spike_mutations_predictor.pkl

Model Info:
  Clades: ['BA.1', 'BA.2', 'B.1.351']
  Performance: R² = 1.000, MAE = 0.009


## 11. Fitness Score Calculation for BLOOM


In [13]:
def calculate_viral_fitness(clade, geo, days_since_start):
    """
    Calculate viral fitness score for BLOOM simulation.

    Args:
        clade: str ('BA.1', 'BA.2', 'B.1.351')
        geo: str (geographic location)
        days_since_start: int (days from 2020-01-28)

    Returns:
        fitness_score: float [0,1] - higher = better fitness
    """

    # Load Categorical features (ESM optional)
    features_cat_input = [[days_since_start, clade, geo]]
    X_cat = oh.transform(features_cat_input)

    # Create placeholder for PCA-transformed ESM embeddings.
    # The xgb_model was trained with pca.n_components_ ESM features.
    # Since no specific sequence is provided here, we'll use zeros.
    # This is a simplification based on ESM's low feature importance (5%).
    X_emb_pca_placeholder = np.zeros((X_cat.shape[0], pca.n_components_))

    # Combine placeholder ESM-PCA features with categorical features
    X_full_input = np.hstack([X_emb_pca_placeholder, X_cat])

    # Predict mutations using the trained XGBoost model
    predicted_mutations = xgb_model.predict(X_full_input)[0]

    # Fitness score: more mutations = higher fitness (adaptation)
    baseline_mutations = y_train.mean()
    fitness = 1.0 - (predicted_mutations - baseline_mutations) / baseline_mutations

    return {
        'clade': clade,
        'geo': geo,
        'days': days_since_start,
        'predicted_mutations': predicted_mutations,
        'fitness_score': max(0.0, fitness)
    }

# Exemple for BLOOM
print("Examples of Fitness Scores for Bloom:")

exemplos = [
    ('BA.2', 'United Kingdom', 750),
    ('BA.2', 'United States', 700),
    ('BA.2', 'Germany', 720),
    ('BA.2', 'Japan', 780),
    ('BA.2', 'Brazil', 760),
    ('BA.2', 'India', 740),
    ('BA.2', 'France', 710),

    ('BA.1', 'USA', 500),
    ('BA.1', 'UK', 480),
    ('BA.1', 'Germany', 520),
    ('BA.1', 'India', 490),
    ('BA.1', 'France', 510),
    ('BA.1', 'Japan', 530),
    ('BA.1', 'Brazil', 470),

    ('B.1.351', 'South Africa', 300),
    ('B.1.351', 'Portugal', 280),
    ('B.1.351', 'UK', 250),
    ]

for clade, geo, days in exemplos:
    result = calculate_viral_fitness(clade, geo, days)
    print(f"  {clade:<8} {geo:<15} day {days:3d} → muts={result['predicted_mutations']:.1f}, fitness={result['fitness_score']:.3f}")

Examples of Fitness Scores for Bloom:
  BA.2     United Kingdom  day 750 → muts=31.0, fitness=0.875
  BA.2     United States   day 700 → muts=31.0, fitness=0.875
  BA.2     Germany         day 720 → muts=31.0, fitness=0.875
  BA.2     Japan           day 780 → muts=31.0, fitness=0.875
  BA.2     Brazil          day 760 → muts=31.0, fitness=0.875
  BA.2     India           day 740 → muts=31.0, fitness=0.875
  BA.2     France          day 710 → muts=31.0, fitness=0.875
  BA.1     USA             day 500 → muts=36.0, fitness=0.692
  BA.1     UK              day 480 → muts=36.0, fitness=0.693
  BA.1     Germany         day 520 → muts=36.0, fitness=0.693
  BA.1     India           day 490 → muts=36.0, fitness=0.693
  BA.1     France          day 510 → muts=36.0, fitness=0.693
  BA.1     Japan           day 530 → muts=36.0, fitness=0.693
  BA.1     Brazil          day 470 → muts=36.0, fitness=0.693
  B.1.351  South Africa    day 300 → muts=10.0, fitness=1.637
  B.1.351  Portugal        day 2

# Protein Language Model (PLM) Results Summary

## Objective

Evaluate whether **protein sequence embeddings from ESM-2 (a Protein Language Model)** improve prediction of SARS-CoV-2 spike mutation counts compared to epidemiological and categorical features.

---

## Data & Modeling Setup

* **Sequences analyzed:** 3,500 spike proteins (2020–2023)
* **Valid after cleaning:** 1,857
* **Target variable:** Number of spike mutations relative to Wuhan-Hu-1 (range 1–42)
* **Feature groups:**
  1. **PLM features:** ESM-2 embeddings (1,280 dimensions → PCA → 2 components)
  2. **Categorical features:** Variant clade, geographic location, time (one-hot encoded)

A temporal split (80% train / 20% test) was used to reflect real-world forecasting.

---

## Distribution Shift Discovery

The test set initially contained a **new variant (XBB.1.5)** with mutation counts much higher than any seen in training data.
This caused severe performance collapse (negative R²), revealing that:

> **The model cannot extrapolate to unseen evolutionary regimes.**

To obtain meaningful evaluation, training and testing were restricted to **shared clades** only (BA.1, BA.2, B.1.351).

---

## Final Performance (Known Variants Only)

| Metric   | Result | Meaning                               |
| -------- | ------ | ------------------------------------- |
| CV R²    | ~0.55  | Reasonable temporal generalization    |
| Train R² | 1.00   | Perfect fit on training data          |
| Test R²  | 1.00   | Accurate predictions for known clades |
| Test MAE | ~0.01  | Extremely small error                 |

**Interpretation:**
The model performs very well **within the distribution of known variants**, but this does not imply predictive ability for new variants.

---

## PLM Contribution (Key Result)

Feature importance analysis shows:

| Feature Type                             | Contribution |
| ---------------------------------------- | ------------ |
| **Categorical (clade, geography, time)** | **~95%**     |
| **PLM embeddings (ESM-2)**               | **~5%**      |

### What this means

* The model mainly learns **variant identity and epidemiological context**.
* **PLM-derived sequence representations play a minor role.**
* Mutation burden is better explained by **evolutionary history and lineage**, not by general protein structural patterns alone.

---

## Biological Interpretation

PLMs capture **biophysical and structural properties** of proteins, but spike mutation accumulation is driven primarily by:

* Selection pressure
* Immune escape dynamics
* Transmission history
* Lineage-specific evolution

Therefore:

> **Evolutionary context dominates over raw protein sequence features for this task.**

---


## Core Conclusion

**Protein Language Models provide useful biochemical representations, but in this study they explain only a small fraction of variance in spike mutation counts. Variant lineage and epidemiological factors are the dominant drivers.**



## Computational Environment

All experiments in this notebook were executed in **Google Colab** using **GPU acceleration**, enabling efficient processing of ESM-2 protein embeddings and model training. GPU usage was particularly important for handling high-dimensional PLM representations and reducing embedding and training time.


